# Visual Product Recommendation System
## End-to-End Pipeline Walkthrough

**Dataset:** Fashion Product Images (Kaggle)  
**Model:** ResNet50 + Siamese Network (Triplet Loss)  
**Author:** MCA II — MIT World Peace University, Pune

---
### Table of Contents
1. Setup & Imports
2. Dataset Preparation
3. Feature Extraction (Baseline ResNet50)
4. Baseline Similarity Search
5. Transfer Learning (Fine-tuning)
6. Siamese Network Training
7. Evaluation: Precision@K & Recall@K
8. Visual Results Comparison

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'CPU only')

## 2. Dataset Preparation

Subset: 5–8 categories, ~250 images each

In [ ]:
from dataset_prep import main as prep_dataset, get_class_names

# Run this once after downloading the Kaggle dataset
# prep_dataset()

# Verify
subset_dir = Path('../data/subset')
if subset_dir.exists():
    for split in ('train', 'val'):
        classes = list((subset_dir / split).iterdir())
        total   = sum(len(list(c.glob('*.jpg'))) for c in classes)
        print(f'{split}: {len(classes)} classes, {total} images')
else:
    print('Run prep_dataset() after downloading dataset')

## 3. Feature Extraction — Baseline ResNet50

In [ ]:
from feature_extractor import build_feature_model, extract_embeddings, save_embeddings

model = build_feature_model()
print('Embedding output shape:', model.output_shape)
model.summary()

In [ ]:
# Extract for train split (run once)
# emb, labels, paths, classes = extract_embeddings(model, 'train')
# save_embeddings(emb, labels, paths, classes, 'train')
# print('Saved embeddings shape:', emb.shape)

# Load pre-computed
from feature_extractor import load_embeddings
try:
    emb, labels, paths, class_names = load_embeddings('train')
    print(f'Loaded: {emb.shape}  |  classes: {class_names}')
except FileNotFoundError:
    print('Run extract_embeddings first')

## 4. Baseline Similarity Search

In [ ]:
from similarity_search import SimilaritySearcher, visualise_results

try:
    searcher = SimilaritySearcher(split='train')

    # Use first image in index as test query
    query_path = searcher.paths[0]
    results = searcher.search(query_path, k=5)

    print(f'Query: {query_path}')
    for r in results:
        print(f"  #{r['rank']}  {r['class_name']:<20} score={r['score']:.4f}")

    # Visualise
    fig = plt.figure(figsize=(18, 4))
    imgs = [query_path] + [r['path'] for r in results]
    for i, p in enumerate(imgs):
        ax = fig.add_subplot(1, len(imgs), i+1)
        try:
            ax.imshow(mpimg.imread(p))
        except Exception:
            pass
        ax.set_title('Query' if i==0 else f"#{i} {results[i-1]['class_name']}\n{results[i-1]['score']:.3f}", fontsize=8)
        ax.axis('off')
    plt.suptitle('Baseline Retrieval (ResNet50)', fontweight='bold')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('Build index first:', e)

## 5. Transfer Learning

In [ ]:
# Uncomment and run to fine-tune:
# from transfer_learning import main as run_transfer_learning
# run_transfer_learning()
print('Fine-tuning: see src/transfer_learning.py')
print('After training, re-run feature extraction with the saved model.')

## 6. Siamese Network Training

In [ ]:
from siamese_network import (
    build_embedding_network, build_siamese_model, triplet_loss, EMBEDDING_DIM
)

emb_net     = build_embedding_network()
siamese_net = build_siamese_model(emb_net)
siamese_net.compile(optimizer='adam', loss=triplet_loss())
print('Siamese network summary:')
siamese_net.summary()

print('\nTriplet loss formula: L = max(0, d(a,p) - d(a,n) + margin)')
print(f'Margin = 0.5  |  Embedding dim = {EMBEDDING_DIM}')

In [ ]:
# Uncomment to train:
# from siamese_network import main as train_siamese
# train_siamese()
print('Training: run `python src/siamese_network.py`')

## 7. Evaluation — Precision@K & Recall@K

In [ ]:
from evaluation import (
    mean_precision_at_k, mean_recall_at_k, measure_inference_time
)

K_VALUES = [1, 3, 5, 10]

try:
    from feature_extractor import load_embeddings
    emb, labels, paths, class_names = load_embeddings('train')

    print(f'\n{"K":<6} {"Precision@K":>14} {"Recall@K":>12}')
    print('─' * 36)
    p_scores, r_scores = [], []
    for k in K_VALUES:
        p = mean_precision_at_k(emb, labels, k=k, n_queries=50)
        r = mean_recall_at_k(emb, labels, k=k, n_queries=50)
        p_scores.append(p)
        r_scores.append(r)
        print(f'K={k:<4} {p:>14.4f} {r:>12.4f}')

    ms = measure_inference_time(emb, n_queries=30)
    print(f'\nAvg inference time (numpy): {ms:.2f} ms')

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(K_VALUES, p_scores, 'o-', color='#5533aa', linewidth=2)
    ax1.set_title('Precision@K (Baseline)', fontweight='bold')
    ax1.set_xlabel('K'); ax1.set_ylabel('Score'); ax1.set_ylim(0, 1)
    ax2.plot(K_VALUES, r_scores, 's-', color='#d4537e', linewidth=2)
    ax2.set_title('Recall@K (Baseline)', fontweight='bold')
    ax2.set_xlabel('K'); ax2.set_ylabel('Score'); ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('../results/eval_baseline.png', dpi=150)
    plt.show()
    print('Plot saved → results/eval_baseline.png')

except FileNotFoundError as e:
    print('Build embeddings first:', e)

## 8. Model Comparison — Baseline vs Siamese

In [ ]:
# Compare embedding spaces via PCA
from utils import pca_2d
import matplotlib.pyplot as plt

try:
    emb_base, labels_base, _, class_names = load_embeddings('train')
    proj = pca_2d(emb_base[:500])

    palette = plt.cm.tab10(np.linspace(0, 1, len(class_names)))
    fig, ax = plt.subplots(figsize=(8, 6))
    for ci, cls in enumerate(class_names):
        mask = labels_base[:500] == ci
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   label=cls, color=palette[ci], alpha=0.7, s=25)
    ax.set_title('Baseline Embedding Space (PCA 2D)', fontweight='bold')
    ax.legend(fontsize=8, loc='best')
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    plt.tight_layout()
    plt.savefig('../results/pca_baseline.png', dpi=150)
    plt.show()
    print('Saved → results/pca_baseline.png')

except Exception as e:
    print(e)

---
## Summary

| Step | Module | Key Output |
|------|--------|------------|
| 1. Dataset | `dataset_prep.py` | `data/subset/{train,val}/{class}/` |
| 2. Feature extract | `feature_extractor.py` | `results/embeddings_train.npz` |
| 3. Baseline search | `similarity_search.py` | Top-K cosine retrieval |
| 4. Transfer learning | `transfer_learning.py` | `models/resnet50_finetuned.h5` |
| 5. Siamese | `siamese_network.py` | `models/siamese_embedding_net.h5` |
| 6. Evaluation | `evaluation.py` | Precision@K, Recall@K plots |
| 7. UI | `app/streamlit_app.py` | Interactive search interface |

Launch the app: `streamlit run app/streamlit_app.py`